# 3-12 重命名索引去重与筛选

本节整理表格清洗中常用的结构操作：重命名、设置与重置索引、去重、删除和集合筛选。

> 约定：使用 `pd` 作为 Pandas 的别名，使用 `np` 作为 NumPy 的别名。本节可独立从上到下运行。

In [ ]:
import pandas as pd
import numpy as np

## 1. 准备示例表格

In [ ]:
df = pd.DataFrame({
    "company": ["B", "A", "B", "B"],
    "gender": ["female", "female", "male", "male"],
    "num": [40, 31, 28, 28]
})
df

## 2. rename：修改列名与行标签

`columns` 和 `index` 都接收“旧名称 → 新名称”的字典。这里把 `num` 改成 `age`，将行标签 0～3 改成 A～D。

In [ ]:
renamed = df.rename(columns={"num": "age"}, index={0: "A", 1: "B", 2: "C", 3: "D"})
renamed

`rename()` 默认返回新对象。使用赋值保存结果，便于跟踪变量；若使用 `inplace=True`，方法返回 `None`，不要将其赋给表格变量。

## 3. set_index 与 reset_index

### 3.1 把字段设为索引

多个字段可以形成多层索引。默认把这些字段从普通列中移除。

In [ ]:
indexed = renamed.set_index(["company", "gender"])
indexed

`company=B, gender=male` 在本例重复出现。索引不强制唯一；设为索引并不等于去重。

### 3.2 重置索引

| 参数 | 原索引的处理 |
|---|---|
| `drop=False`（默认） | 原索引恢复为普通列 |
| `drop=True` | 丢弃原索引 |

两种情况都会重新生成默认的整数索引。

In [ ]:
reset = indexed.reset_index(drop=False)
reset

In [ ]:
indexed.reset_index(drop=True)

## 4. drop_duplicates：删除重复记录

默认按所有列判断是否重复，并保留第一次出现的记录。

In [ ]:
reset.drop_duplicates()

`subset` 指定用哪些列判断重复；`keep="last"` 保留最后一条，`keep=False` 则删除该键的所有重复记录。

In [ ]:
reset.drop_duplicates(subset=["company"], keep="last")

结果保留公司 A 的索引 1 与公司 B 的索引 3。这是按出现顺序保留最后一条，不代表最大年龄或最新日期；如果有这类需求，应先排序。

In [ ]:
reset.drop_duplicates(subset=["company"], keep=False)

只剩公司 A，因为公司 B 出现过多次。

## 5. drop：删除指定行或列

`columns` 指定列标签，`index` 指定行标签；这里删除 `age` 列和标签为 0 的行。

In [ ]:
reset.drop(columns="age", index=0)

默认返回新表，`reset` 仍保留完整数据。`index=0` 表示行标签 0，不能在任意索引中理解为“第一行”。

## 6. isin：判断是否属于一组值

`isin()` 返回布尔值，常与 `loc` 组合筛选。

In [ ]:
reset.loc[reset["gender"].isin(["male"])]

In [ ]:
reset.loc[reset["company"].isin(["A"])]

传入多个候选值时，表示“属于其中任意一个”；在条件前加 `~` 表示取反。

In [ ]:
reset.loc[~reset["company"].isin(["A"])]

## 7. 要点总结

- `rename` 修改标签；`set_index` 用字段建立索引；`reset_index` 恢复默认索引。
- `reset_index(drop=True)` 会丢弃原索引，操作前应确认是否还需要这些字段。
- `drop_duplicates` 的 `subset` 决定比较字段，`keep` 决定保留哪条。
- `drop` 按标签删除，`isin` 生成集合成员关系的布尔条件。